In [ ]:
%%bash
set -e
CODE=$(dirname "$(find /kaggle/input -name train.py -not -path '*t3*' | head -1)")
DATA=$(dirname "$(find /kaggle/input -name train_images.npy | head -1)")
T3=$(dirname "$(find /kaggle/input -name t4_filter_fit.npy | head -1)")
echo "code=$CODE data=$DATA t3=$T3"; nvidia-smi -L
rm -rf /kaggle/working/cv && cp -r "$CODE" /kaggle/working/cv
ln -sfn "$DATA" /kaggle/working/data
mkdir -p /kaggle/working/t4_inputs && cp "$T3/t4_filter_fit.npy" "$T3/t4_relabel.npy" /kaggle/working/t4_inputs/
python -c "import numpy as np; f=np.load('/kaggle/working/t4_inputs/t4_filter_fit.npy'); r=np.load('/kaggle/working/t4_inputs/t4_relabel.npy'); print('filter', f.shape, 'relabel', r.shape)"

In [ ]:
%%bash
# T4 filter + relabel (noisy labels, Task 3 outputs only; no clean labels). Same setup as all other runs.
cd /kaggle/working/cv
F="--fit_idx ../t4_inputs/t4_filter_fit.npy"; R="--relabel ../t4_inputs/t4_relabel.npy"
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  CUDA_VISIBLE_DEVICES=0 python train.py --data ../data --out ../outputs/t4_filter  --label_col noisy_label $F > ../t4_filter.log 2>&1 &
  CUDA_VISIBLE_DEVICES=1 python train.py --data ../data --out ../outputs/t4_relabel --label_col noisy_label $R > ../t4_relabel.log 2>&1 &
  wait
else
  python train.py --data ../data --out ../outputs/t4_filter  --label_col noisy_label $F > ../t4_filter.log 2>&1
  python train.py --data ../data --out ../outputs/t4_relabel --label_col noisy_label $R > ../t4_relabel.log 2>&1
fi
tail -n 1 ../t4_filter.log ../t4_relabel.log

In [ ]:
%%bash
cd /kaggle/working/cv
for r in t4_filter t4_relabel; do
  python make_preds.py --run ../outputs/$r --epoch final --out ../predictions/${r}_final.csv
  python make_preds.py --run ../outputs/$r --epoch best  --out ../predictions/${r}_best.csv
done
rm -f /kaggle/working/data